In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS SILVER;

CREATE SCHEMA IF NOT EXISTS GOLD;

In [0]:
%sql
CREATE TABLE silver.ODS_MEMBER (
    MEMBER_HIST_KEY BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    MEMBER_ID STRING NOT NULL,
    FIRST_NAME STRING,
    LAST_NAME STRING,
    DOB DATE,
    GENDER STRING,
    PLAN_ID STRING,
    EMAIL STRING,
    PHONE STRING,
    ADDRESS_LINE1 STRING,
    ADDRESS_LINE2 STRING,
    CITY STRING,
    STATE STRING,
    ZIP_CODE STRING,
    COUNTRY STRING,
    SRC_CREATED_DATETIME TIMESTAMP,
    FULL_NAME STRING,
    AGE INT,
    AGE_GROUP STRING,
    HASH_VALUE STRING,
    LOAD_DATE TIMESTAMP,
    IS_CURRENT STRING,
    EFFECTIVE_DATE TIMESTAMP,
    END_DATE TIMESTAMP,
    CONSTRAINT PK_ODS_MEMBER PRIMARY KEY (MEMBER_HIST_KEY)
) USING DELTA;


In [0]:
%sql
CREATE TABLE databricks_ods.silver.ODS_PROVIDER (
    PROVIDER_HIST_KEY BIGINT GENERATED ALWAYS AS IDENTITY,
    PROVIDER_ID STRING NOT NULL,
    PROVIDER_NAME STRING,
    SPECIALITY STRING,
    STATUS STRING,
    EMAIL STRING,
    PHONE STRING,
    ADDRESS_LINE1 STRING,
    ADDRESS_LINE2 STRING,
    CITY STRING,
    STATE STRING,
    ZIP_CODE STRING,
    COUNTRY STRING,
    SRC_CREATED_DATETIME TIMESTAMP,
    HASH_VALUE STRING,
    LOAD_DATE TIMESTAMP,
    IS_CURRENT STRING,
    EFFECTIVE_DATE TIMESTAMP,
    END_DATE TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
DROP TABLE IF EXISTS databricks_ods.silver.ods_claim;

CREATE TABLE databricks_ods.silver.ods_claim (
    CLAIM_ID             BIGINT,
    MEMBER_ID            STRING,
    PROVIDER_ID          STRING,
    CLAIM_DATE           DATE,
    RECEIVED_DATE        DATE,
    PROCESSED_DATE       DATE,
    PAID_DATE            DATE,
    CLAIM_AMOUNT         DECIMAL(18,2),
    CLAIM_STATUS         STRING,
    DIAGNOSIS_CODE       STRING,
    PROCEDURE_CODE       STRING,
    SRC_CREATED_DATETIME TIMESTAMP,   

    CLAIM_YEAR           INT,
    CLAIM_MONTH          INT,
    CLAIM_AGE_DAYS       INT,
    PROCESSING_TIME_DAYS INT,
    PAYMENT_LAG_DAYS     INT,
    CLAIM_BUCKET         STRING,
    STATUS_FLAG          STRING,

    HASH_VALUE           STRING,
    LOAD_DATE            TIMESTAMP,
    IS_CURRENT           STRING,
    EFFECTIVE_DATE       TIMESTAMP,
    END_DATE             TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
DROP TABLE IF EXISTS databricks_ods.silver.ods_payment;

CREATE TABLE databricks_ods.silver.ods_payment (
    PAYMENT_ID           STRING,
    CLAIM_ID             STRING,
    PAYMENT_DATE         DATE,
    PAYMENT_AMOUNT       DECIMAL(18,2),
    PAYMENT_STATUS       STRING,
    SRC_CREATED_DATETIME TIMESTAMP,

    -- Derived fields
    PAYMENT_YEAR         INT,
    PAYMENT_MONTH        INT,
    PAYMENT_DELAY_DAYS   INT,
    STATUS_FLAG          STRING,

    -- SCD2 / Audit fields
    HASH_VALUE           STRING,
    LOAD_DATE            TIMESTAMP,
    IS_CURRENT           STRING,
    EFFECTIVE_DATE       TIMESTAMP,
    END_DATE             TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
DROP TABLE IF EXISTS databricks_ods.gold.dim_member;

CREATE TABLE databricks_ods.gold.dim_member (
    MEMBER_ID   STRING NOT NULL,
    FULL_NAME   STRING,
    DOB         DATE,
    GENDER      STRING,
    PLAN_ID     STRING,
    AGE         INT,
    AGE_GROUP   STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
DROP TABLE IF EXISTS databricks_ods.gold.dim_provider;

CREATE TABLE databricks_ods.gold.dim_provider (
    PROVIDER_ID   STRING NOT NULL,
    PROVIDER_NAME STRING,
    SPECIALITY    STRING,
    STATUS        STRING,
    CITY          STRING,
    STATE         STRING,
    COUNTRY       STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
DROP TABLE IF EXISTS databricks_ods.gold.dim_date;

CREATE TABLE databricks_ods.gold.dim_date (
    DATE         DATE,
    DATE_ID      STRING,
    YEAR         INT,
    MONTH        INT,
    DAY          INT,
    WEEKDAY      STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
DROP TABLE IF EXISTS databricks_ods.gold.fact_claim;

CREATE TABLE databricks_ods.gold.fact_claim (
    CLAIM_ID             BIGINT NOT NULL,
    MEMBER_ID            STRING,
    PROVIDER_ID          STRING,
    DATE_ID              STRING,
    CLAIM_AMOUNT         DECIMAL(18,2),
    CLAIM_YEAR           INT,
    CLAIM_MONTH          INT,
    CLAIM_AGE_DAYS       INT,
    PROCESSING_TIME_DAYS INT,
    PAYMENT_LAG_DAYS     INT,
    CLAIM_BUCKET         STRING,
    STATUS_FLAG          STRING
)
USING DELTA

TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
DROP TABLE IF EXISTS databricks_ods.gold.fact_payment;

CREATE TABLE databricks_ods.gold.fact_payment (
    PAYMENT_ID         STRING NOT NULL,
    CLAIM_ID           BIGINT,
    DATE_ID            STRING,
    PAYMENT_AMOUNT     DECIMAL(18,2),
    PAYMENT_YEAR       INT,
    PAYMENT_MONTH      INT,
    PAYMENT_DELAY_DAYS INT,
    STATUS_FLAG        STRING
)
USING DELTA

TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


In [0]:
%sql
SHOW TABLES IN gold